# Enhanced World Model - Google Colab Training

Train world models for reinforcement learning on Google Colab with GPU acceleration.

**Features:**
- Vision (VQ-VAE) + Memory (LSTM/Transformer) + Controller (PPO)
- Supports both vector-based (CartPole) and image-based (CarRacing) environments
- Automatic exploration annealing for stable training

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the repository
!git clone https://github.com/Larwive/Enhanced-World-Model.git
%cd Enhanced-World-Model

In [ ]:
# Install dependencies
!pip install gymnasium[all] torch torchvision tensorboard matplotlib numpy psutil

In [ ]:
# Add src to path
import sys

sys.path.insert(0, "src")

# Verify imports
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Training Configuration
CONFIG = {
    # Environment
    "env_name": "CartPole-v1",  # Options: "CartPole-v1", "CarRacing-v3", "LunarLander-v3"
    # Model architecture
    "vision": "Identity",  # "Identity" for vector envs, "VQ_VAE" for image envs
    "memory": "LSTMMemory",  # "LSTMMemory" or "TemporalTransformer"
    "controller": "auto",  # "auto" selects based on action space
    # Training
    "max_epochs": 500,
    "learning_rate": 3e-4,
    "num_envs": 32,  # Parallel environments (increase for GPU)
    # PPO hyperparameters
    "rollout_steps": 128,
    "num_ppo_epochs": 4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_epsilon": 0.2,
    # Exploration annealing
    "entropy_start": 0.01,
    "entropy_end": 0.001,
    "temp_start": 1.0,
    "temp_end": 0.1,
    "anneal_steps": 50000,
}

# Auto-detect settings for common environments
if "CarRacing" in CONFIG["env_name"] or "Atari" in CONFIG["env_name"]:
    CONFIG["vision"] = "VQ_VAE"
    CONFIG["num_envs"] = 8  # Image envs need fewer parallel envs
    print("Detected image-based environment, using VQ_VAE vision")

print(f"Configuration: {CONFIG}")

## 3. Initialize Model and Environment

In [ ]:
import gymnasium as gym
import numpy as np
import torch

import controller
import memory
import reward_predictor

# Import model components
import vision
from utils.registry import discover_modules
from WorldModel import WorldModel

# Discover available models
VISION_REGISTRY = discover_modules(vision)
MEMORY_REGISTRY = discover_modules(memory)
CONTROLLER_REGISTRY = discover_modules(controller)
REWARD_PREDICTOR_REGISTRY = discover_modules(reward_predictor)

print("Available models:")
print(f"  Vision: {list(VISION_REGISTRY.keys())}")
print(f"  Memory: {list(MEMORY_REGISTRY.keys())}")
print(f"  Controller: {list(CONTROLLER_REGISTRY.keys())}")

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

# Create vectorized environment
envs = gym.make_vec(CONFIG["env_name"], num_envs=CONFIG["num_envs"], render_mode="rgb_array")

obs_space = envs.single_observation_space
action_space = envs.single_action_space
is_image_based = len(obs_space.shape) == 3
is_discrete_action = isinstance(action_space, gym.spaces.Discrete)

print(f"Environment: {CONFIG['env_name']}")
print(f"  Observation space: {obs_space}")
print(f"  Action space: {action_space}")
print(f"  Image-based: {is_image_based}")
print(f"  Discrete actions: {is_discrete_action}")

In [ ]:
# Configure model dimensions
if is_image_based:
    obs_shape = obs_space.shape
    input_shape = (obs_shape[2], obs_shape[0], obs_shape[1])  # CHW
    vision_args = {"output_dim": input_shape[0], "embed_dim": 64}
else:
    input_shape = obs_space.shape
    vision_args = {"embed_dim": obs_space.shape[0]}

if is_discrete_action:
    action_dim = action_space.n
else:
    action_dim = action_space.shape[0]

memory_args = {
    "d_model": 128,
    "latent_dim": vision_args["embed_dim"],
    "action_dim": action_dim,
    "nhead": 8,
}
controller_args = {"action_dim": action_dim}

# Auto-select controller
if CONFIG["controller"] == "auto":
    if is_discrete_action:
        controller_name = "DiscreteModelPredictiveController"
    else:
        controller_name = "ContinuousModelPredictiveController"
else:
    controller_name = CONFIG["controller"]

print("\nModel configuration:")
print(f"  Vision: {CONFIG['vision']} - {vision_args}")
print(f"  Memory: {CONFIG['memory']} - {memory_args}")
print(f"  Controller: {controller_name} - {controller_args}")

In [ ]:
# Create WorldModel
world_model = WorldModel(
    vision_model=VISION_REGISTRY[CONFIG["vision"]],
    memory_model=MEMORY_REGISTRY[CONFIG["memory"]],
    controller_model=CONTROLLER_REGISTRY[controller_name],
    input_shape=input_shape,
    vision_args=vision_args,
    memory_args=memory_args,
    controller_args=controller_args,
).to(device)

# Add reward predictor
world_model.set_reward_predictor(REWARD_PREDICTOR_REGISTRY["LinearPredictor"])

# Count parameters
total_params = sum(p.numel() for p in world_model.parameters())
trainable_params = sum(p.numel() for p in world_model.parameters() if p.requires_grad)
print("\nModel created!")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

## 4. Training

In [ ]:
# Start TensorBoard (run in separate cell to view)
%load_ext tensorboard
%tensorboard --logdir runs/

In [ ]:
import os
from datetime import datetime

from train import train

# Create save directory
save_dir = "./saved_models/"
os.makedirs(save_dir, exist_ok=True)

print("Starting training...")
print(f"  Environment: {CONFIG['env_name']}")
print(f"  Parallel envs: {CONFIG['num_envs']}")
print(f"  Max epochs: {CONFIG['max_epochs']}")
print(f"  Device: {device}")
print()

# Train!
train(
    world_model,
    envs,
    max_iter=CONFIG["max_epochs"],
    device=device,
    learning_rate=CONFIG["learning_rate"],
    save_path=save_dir,
    render_mode="rgb_array",
    # PPO hyperparameters
    rollout_steps=CONFIG["rollout_steps"],
    num_epochs=CONFIG["num_ppo_epochs"],
    gamma=CONFIG["gamma"],
    gae_lambda=CONFIG["gae_lambda"],
    clip_epsilon=CONFIG["clip_epsilon"],
    # Exploration annealing
    entropy_coeff_start=CONFIG["entropy_start"],
    entropy_coeff_end=CONFIG["entropy_end"],
    temp_start=CONFIG["temp_start"],
    temp_end=CONFIG["temp_end"],
    anneal_steps=CONFIG["anneal_steps"],
)

In [ ]:
# Save final model
save_name = f"{save_dir}{CONFIG['env_name']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt"
world_model.save(save_name, obs_space, action_space)
print(f"Model saved to: {save_name}")

## 5. Evaluation

In [ ]:
import time

import matplotlib.pyplot as plt
from IPython.display import clear_output


def evaluate_model(model, env_name, num_episodes=5, render=True):
    """Evaluate the trained model."""
    env = gym.make(env_name, render_mode="rgb_array")
    model.eval()

    episode_rewards = []
    episode_lengths = []

    for ep in range(num_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0
        length = 0
        frames = []

        # Reset model memory
        model.a_prev = None
        if hasattr(model.memory, "reset"):
            model.memory.reset()

        while not done:
            # Prepare observation
            if is_image_based:
                obs_tensor = torch.from_numpy(obs).float().permute(2, 0, 1).unsqueeze(0) / 255.0
            else:
                obs_tensor = torch.from_numpy(obs).float().unsqueeze(0)
            obs_tensor = obs_tensor.to(device)

            # Get action
            with torch.no_grad():
                action = model(obs_tensor, action_space, is_image_based, return_losses=False)
                action = action.cpu().numpy()[0]

            # Step environment
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            length += 1

            if render:
                frames.append(env.render())

        episode_rewards.append(total_reward)
        episode_lengths.append(length)
        print(f"Episode {ep+1}: Reward = {total_reward:.1f}, Length = {length}")

        # Show last episode
        if render and ep == num_episodes - 1 and frames:
            fig, axes = plt.subplots(1, min(5, len(frames)), figsize=(15, 3))
            indices = np.linspace(0, len(frames) - 1, min(5, len(frames)), dtype=int)
            for i, idx in enumerate(indices):
                axes[i].imshow(frames[idx])
                axes[i].set_title(f"Step {idx}")
                axes[i].axis("off")
            plt.suptitle(f"Episode {ep+1} - Reward: {total_reward:.1f}")
            plt.tight_layout()
            plt.show()

    env.close()

    print("\nEvaluation Summary:")
    print(f"  Mean Reward: {np.mean(episode_rewards):.1f} +/- {np.std(episode_rewards):.1f}")
    print(f"  Mean Length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")

    return episode_rewards, episode_lengths

In [ ]:
# Evaluate the trained model
rewards, lengths = evaluate_model(world_model, CONFIG["env_name"], num_episodes=10)

## 6. Download Model

In [ ]:
# List saved models
import os

print("Saved models:")
for f in os.listdir(save_dir):
    if f.endswith(".pt"):
        size = os.path.getsize(os.path.join(save_dir, f)) / 1024 / 1024
        print(f"  {f} ({size:.1f} MB)")

In [ ]:
# Download model (Google Colab)
from google.colab import files

# Download the most recent model
model_files = [f for f in os.listdir(save_dir) if f.endswith(".pt")]
if model_files:
    latest_model = sorted(model_files)[-1]
    files.download(os.path.join(save_dir, latest_model))
    print(f"Downloaded: {latest_model}")
else:
    print("No models found to download")

## 7. Load and Continue Training (Optional)

In [ ]:
# Upload a model file (Google Colab)
# from google.colab import files
# uploaded = files.upload()
# model_path = list(uploaded.keys())[0]

In [ ]:
# Load a saved model
# model_path = "./saved_models/your_model.pt"
# world_model.load(model_path, obs_space, action_space)
# print(f"Loaded model from {model_path}")
# print(f"  Iterations: {world_model.iter_num}")
# print(f"  Experiments: {world_model.nb_experiments}")

---

## Quick Configs for Different Environments

### CartPole (Fast, good for testing)
```python
CONFIG = {
    "env_name": "CartPole-v1",
    "vision": "Identity",
    "memory": "LSTMMemory",
    "num_envs": 64,
    "max_epochs": 500,
}
```

### LunarLander (Medium difficulty)
```python
CONFIG = {
    "env_name": "LunarLander-v3",
    "vision": "Identity",
    "memory": "LSTMMemory",
    "num_envs": 32,
    "max_epochs": 1000,
}
```

### CarRacing (Image-based, GPU recommended)
```python
CONFIG = {
    "env_name": "CarRacing-v3",
    "vision": "VQ_VAE",
    "memory": "LSTMMemory",
    "num_envs": 8,
    "max_epochs": 5000,
}
```